In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"
print("✅ API Key configured")

✅ API Key configured


In [2]:
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from datetime import datetime
print("✅ All imports successful")

c:\Users\shivam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ All imports successful


In [3]:
from groq import Groq


client = Groq(api_key=os.environ["GROQ_API_KEY"])
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello!"}]
)
print("✅ Key check:", response.choices[0].message.content)


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    api_key=os.environ["GROQ_API_KEY"]
)
print("✅ Groq LLM ready")

✅ Key check: Hello. It's nice to meet you. Is there something I can help you with, or would you like to chat?
✅ Groq LLM ready


In [4]:
from langchain_core.tools import StructuredTool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
wikipedia = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)  # reduced

def web_search(query: str) -> str:
    """Search internet for current information and facts."""
    result = search.run(query)
    return result[:500]  

def wiki_search(query: str) -> str:
    """Look up background information from Wikipedia."""
    result = wikipedia.run(query)
    return result[:500]  

web_search_tool = StructuredTool.from_function(
    func=web_search,
    name="web_search",
    description="Search internet for current information and facts."
)

wiki_search_tool = StructuredTool.from_function(
    func=wiki_search,
    name="wiki_search",
    description="Look up background information from Wikipedia."
)

tools = [web_search_tool, wiki_search_tool]
print("✅ Both tools ready")

✅ Both tools ready


In [5]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=tools
)
print("✅ ReAct Agent ready")

✅ ReAct Agent ready


C:\Users\shivam\AppData\Local\Temp\ipykernel_29208\1785437117.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [6]:
def run_research_agent(topic: str) -> dict:
    print(f"\n{'='*65}")
    print(f"  🔍 RESEARCHING: {topic}")
    print(f"{'='*65}\n")

    
    research_query = (
        f"Research '{topic}'. Find: overview, applications, "
        f"key statistics, challenges, and future trends. "
        f"Use web_search and wiki_search tools. Be concise."
    )

    result = agent.invoke({"messages": [("user", research_query)]})
    research_findings = result["messages"][-1].content

    print(f"\n{'='*65}")
    print("  📝 GENERATING STRUCTURED REPORT...")
    print(f"{'='*65}\n")

    report = generate_report(topic, research_findings)
    print("\n✅ Report generation complete!")

    return {
        "topic": topic,
        "research_findings": research_findings,
        "report": report,
        "steps_taken": len(result["messages"])
    }

print("✅ Functions ready")

✅ Functions ready


In [69]:
output_1 = run_research_agent("Impact of AI in Healthcare")
print(output_1["report"])
save_report(output_1, "sample_output_1_ai_healthcare.txt")


  🔍 RESEARCHING: Impact of AI in Healthcare


  📝 GENERATING STRUCTURED REPORT...


✅ Report generation complete!
═══════════════════════════════════════════════════════════════
                          COVER PAGE
═══════════════════════════════════════════════════════════════
Report Title : Revolutionizing Healthcare: The Impact of Artificial Intelligence
Topic        : Impact of AI in Healthcare
Prepared by  : Autonomous Research Agent (LangChain + Groq)
Date         : March 25, 2026
Tools Used   : DuckDuckGo Web Search, Wikipedia
Agent Type   : ReAct (LangGraph)
═══════════════════════════════════════════════════════════════

1. INTRODUCTION
The integration of Artificial Intelligence (AI) in healthcare has been a topic of significant interest and research in recent years. The potential of AI to transform the healthcare industry is vast, with applications in diagnosis, treatment, and patient care. This report aims to provide an in-depth analysis of the impact of AI in healthcare, i

In [7]:
output_2 = run_research_agent("Renewable Energy and Climate Change")
print(output_2["report"])
save_report(output_2, "sample_output_2_renewable_energy.txt")


  🔍 RESEARCHING: Renewable Energy and Climate Change



BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=web_search{"query": "Renewable Energy and Climate Change overview"}</function>\n'}}